In [6]:
import numpy as np
import matplotlib.pyplot as plt
import time

# фотонов на МэВ
YIELD = 10000 

In [7]:
class PMT:
    def __init__(self, pos, r):
        self.pos = pos
        self.r = r


class Detector:
    def __init__(self, R, N, r, a):
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        self.pmts = []
        self._make_pmts()
        
        # массив позиций для быстрых вычислений
        self.pos_arr = np.array([p.pos for p in self.pmts])
        
        # сетка для поиска (spatial hashing)
        self.cell = 2 * r
        self.grid = {}
        for i, p in enumerate(self.pmts):
            key = tuple((p.pos // self.cell).astype(int))
            self.grid.setdefault(key, []).append(i)
    
    def _make_pmts(self):
        i = np.arange(self.N) + 0.5
        phi = np.arccos(1 - 2*i/self.N)
        theta = np.pi * (1 + 5**0.5) * i
        x = self.R * np.sin(phi) * np.cos(theta)
        y = self.R * np.sin(phi) * np.sin(theta)
        z = self.R * np.cos(phi)
        
        for k in range(self.N):
            self.pmts.append(PMT(np.array([x[k], y[k], z[k]]), self.r))

    def hit_point(self, origin, dirs):
        # dirs: (3, N)
        a = np.sum(dirs**2, axis=0)
        b = 2 * np.dot(origin, dirs)
        c = np.dot(origin, origin) - self.R**2
        
        disc = b**2 - 4*a*c
        valid = disc >= 0
        t = np.full(dirs.shape[1], np.nan)
        t[valid] = (-b[valid] + np.sqrt(disc[valid])) / (2*a[valid])
        
        t[t < 0] = np.nan
        return origin[:, None] + t * dirs
    
    def find_pmt(self, pts):
        # pts: (3, N)
        N = pts.shape[1]
        res = np.full(N, -1, dtype=int)
        
        valid = ~np.isnan(pts[0])
        if np.sum(valid) == 0:
            return res
            
        vp = pts[:, valid]
        
        # ищем по сетке
        for k in range(vp.shape[1]):
            p = vp[:, k]
            key = tuple((p // self.cell).astype(int))
            
            cand = self.grid.get(key, [])
            if not cand:
                continue
            
            # считаем расстояния только до кандидатов
            d = np.linalg.norm(self.pos_arr[cand] - p, axis=1)
            j = np.argmin(d)
            
            if d[j] <= self.r:
                # восстанавливаем индекс в исходном массиве
                orig_i = np.where(valid)[0][k]
                res[orig_i] = cand[j]
                
        return res

In [8]:
class Source:
    def __init__(self, Y=YIELD, L_att=8.0):
        self.Y = Y
        self.L_att = L_att
    
    def n_photons(self, E):
        return np.random.poisson(self.Y * E)
    
    def directions(self, N):
        phi = np.random.uniform(0, 2*np.pi, N)
        cos_t = np.random.uniform(-1, 1, N)
        sin_t = np.sqrt(1 - cos_t**2)
        return np.array([sin_t * np.cos(phi), sin_t * np.sin(phi), cos_t])


def run_event(det, src, E, pos=None, att=True, chunk=2000):
    if pos is None:
        pos = np.zeros(3)
        
    N = src.n_photons(E)
    if N == 0:
        return 0
        
    dirs = src.directions(N)
    total = 0
    
    # обрабатываем пачками, чтобы не забить память
    for i in range(0, N, chunk):
        d_chunk = dirs[:, i:i+chunk]
        
        hits = det.hit_point(pos, d_chunk)
        idx = det.find_pmt(hits)
        
        ok = idx != -1
        if np.sum(ok) == 0:
            continue
            
        h_ok = hits[:, ok]
        
        if att:
            dist = np.linalg.norm(h_ok - pos[:, None], axis=0)
            p = np.exp(-dist / src.L_att)
            survive = np.random.uniform(0, 1, len(p)) < p
            h_ok = h_ok[:, survive]
            
        if h_ok.shape[1] > 0:
            total += np.sum(np.random.poisson(det.a, size=h_ok.shape[1]))
            
    return total

In [9]:
# большой детектор
big_det = Detector(R=15.0, N=2000, r=0.1, a=0.5)
src = Source()

E0 = 1.0
n_ev = 10

start = time.perf_counter()
charges = [run_event(big_det, src, E0) for _ in range(n_ev)]
t_total = time.perf_counter() - start

print(f"Событий: {n_ev}, время: {t_total:.2f} сек")
print(f"Средний заряд: {np.mean(charges):.1f}")

Событий: 10, время: 0.31 сек
Средний заряд: 9.1


In [10]:
def find_pmt_slow(det, pts):
    # полный перебор, как в начале
    res = []
    for k in range(pts.shape[1]):
        p = pts[:, k]
        if np.isnan(p[0]):
            res.append(-1)
            continue
        d = np.linalg.norm(det.pos_arr - p, axis=1)
        j = np.argmin(d)
        if d[j] <= det.r:
            res.append(j)
        else:
            res.append(-1)
    return np.array(res)

# одно событие с малой энергией
E_small = 0.05
dirs_small = src.directions(src.n_photons(E_small))
hits_small = big_det.hit_point(np.zeros(3), dirs_small)

t0 = time.perf_counter()
_ = big_det.find_pmt(hits_small)
t_fast = time.perf_counter() - t0

t0 = time.perf_counter()
_ = find_pmt_slow(big_det, hits_small)
t_slow = time.perf_counter() - t0

print(f"Фотонов: {len(dirs_small[0])}")
print(f"Сетка: {t_fast*1000:.2f} мс")
print(f"Перебор: {t_slow*1000:.2f} мс")

Фотонов: 508
Сетка: 1.95 мс
Перебор: 36.14 мс
